In [3]:
#path configuations

from typing import Optional
from pathlib import Path
import json

def find_repo_root(start: Optional[Path] = None, repo_name: str = "masters_thesis_sdg") -> Path:
    """
    Walk upward from the current working directory until the repository root is found.
    """
    if start is None:
        start = Path.cwd().resolve()

    current = start
    while True:
        if current.name == repo_name:
            return current
        if current.parent == current:
            raise FileNotFoundError(
                f"Could not find repo root '{repo_name}' starting from {start}"
            )
        current = current.parent


REPO_ROOT = find_repo_root()
idx_file_path = REPO_ROOT / "data" / "processed" / "lai2023" / "onuw_transcripts_ready" / "index_cleaned.json"

RESULTS_DIR = REPO_ROOT / "results" / "voting"

llm = "unsloth_gpt-oss-20b"
#llm = "gemma-4-31B-it"

#prompt version used for the voting task
prompt_version = "v3"


In [4]:

# Load the Ground Truth Index
with open(idx_file_path, "r", encoding="utf-8") as f:
    index_data = json.load(f)

ground_truth_lookup = {
    (game["session_name"], game["game_key"]): game
    for game in index_data
}

#Find the Correct Model & Prompt Directory
model_dirs = [d for d in RESULTS_DIR.iterdir() if d.is_dir() and llm in d.name]

if not model_dirs:
    raise FileNotFoundError(f"Could not find a results directory for model '{llm}' in {RESULTS_DIR}")

model_dir = model_dirs[0]
prompt_dir = model_dir / f"prompt_{prompt_version}"

if not prompt_dir.exists():
    raise FileNotFoundError(f"Could not find prompt directory: {prompt_dir}")

print(f"Loading results from: {prompt_dir}\n")

Loading results from: C:\Users\annab\Documents\GitHub\masters_thesis_sdg\results\voting\unsloth_gpt-oss-20b-unsloth-bnb-4bit\prompt_v3



## Accuracy evaluation
First of all, I analyse the accuracy of the model finding the werewolf. This gives me an idea of the difficulty of the task

In [8]:
# --- Evaluate Predictions ---
total_games_evaluated = 0
correct_werewolf_votes = 0
correct_circle_votes = 0  # Track successful circle votes
attempted_circle_votes = 0 # NEW: Track ALL circle votes attempted by the LLM
invalid_votes = 0
failed_parses = 0

# rglob("*.json") recursively finds all JSONs in subfolders.
# This natively grabs files from both /Youtube/ and /Ego4D/ jointly.
result_files = list(prompt_dir.rglob("*.json"))

for res_file in result_files:
    with open(res_file, "r", encoding="utf-8") as f:
        res_data = json.load(f)
        
    session_name = res_data.get("session_name")
    game_key = res_data.get("game_key")
    
    gt_game = ground_truth_lookup.get((session_name, game_key))
    if not gt_game:
        print(f"Warning: Ground truth not found for {session_name} - {game_key}")
        continue
        
    total_games_evaluated += 1
    
    validation_info = res_data.get("validation", {})
    is_valid = validation_info.get("is_valid", False)
    chosen_vote = validation_info.get("chosen_vote")
    
    if not is_valid or not chosen_vote:
        failed_parses += 1
        continue
        
    player_names = gt_game["player_names"]
    
    # 1. Did the LLM vote for a specific player?
    if chosen_vote in player_names:
        player_idx = player_names.index(chosen_vote)
        voted_player_end_role = gt_game["end_roles"][player_idx]
        
        if voted_player_end_role == "Werewolf":
            correct_werewolf_votes += 1
            
    # 2. Did the LLM vote for a Circle Vote? (using .lower() for safety)
    elif chosen_vote.strip().lower() == "circle vote":
        attempted_circle_votes += 1 # The LLM tried to circle vote
        
        # It's only a correct move if there are no werewolves in play
        if "Werewolf" not in gt_game["end_roles"]:
            correct_circle_votes += 1
            
    # 3. If it's not a player and not a circle vote, it's invalid
    else:
        invalid_votes += 1

# --- Print Results ---
if total_games_evaluated > 0:
    valid_game_count = total_games_evaluated - failed_parses - invalid_votes
    
    # Accuracy now combines both types of successful wins for the village
    total_correct = correct_werewolf_votes + correct_circle_votes
    accuracy = (total_correct / valid_game_count) * 100 if valid_game_count > 0 else 0
    
    print("-" * 45)
    print(f"Results: {llm} | Prompt: {prompt_version}")
    print("-" * 45)
    print(f"Total Games Evaluated:     {total_games_evaluated}")
    print(f"Correct Werewolf Votes:    {correct_werewolf_votes}")
    print(f"Correct Circle Votes:      {correct_circle_votes}")
    print(f"Total Attempted Circles:   {attempted_circle_votes}")
    print(f"Failed to Parse Vote:      {failed_parses}")
    print(f"Hallucinated Names:        {invalid_votes}")
    print("-" * 45)
    print(f"Valid Votes Cast:          {valid_game_count}")
    print(f"Total LLM Wins:            {total_correct}")
    print(f"Accuracy (Valid Votes):    {accuracy:.2f}%")
else:
    print("No result files were found or evaluated.")

---------------------------------------------
Results: unsloth_gpt-oss-20b | Prompt: v3
---------------------------------------------
Total Games Evaluated:     191
Correct Werewolf Votes:    65
Correct Circle Votes:      0
Total Attempted Circles:   0
Failed to Parse Vote:      26
Hallucinated Names:        0
---------------------------------------------
Valid Votes Cast:          165
Total LLM Wins:            65
Accuracy (Valid Votes):    39.39%


In [15]:
#human accuracy
from collections import Counter

# --- Calculate Human Village Win Rate ---
total_games_in_index = len(index_data)
valid_human_games = 0
human_correct_werewolf_kills = 0
human_correct_circle_votes = 0
attempted_circle_votes = 0

for game in index_data:
    raw_votes = game.get("voting_outcome", [])
    
    # FIX: Skip games where human votes are missing or marked as "N/A"
    if not raw_votes or "N/A" in raw_votes or "NA" in raw_votes or raw_votes == "N/A":
        continue
        
    # Convert valid votes to integers
    votes = [int(v) for v in raw_votes]
    end_roles = game.get("end_roles", [])
    
    # Track that we have a valid game to evaluate
    valid_human_games += 1
    
    # Tally up the votes received by each player index
    vote_counts = Counter(votes)
    
    if not vote_counts:
        continue
        
    # Find the maximum number of votes received by anyone
    max_votes = max(vote_counts.values())
    
    # ONUW Rule: To be eliminated, a player must receive at least 2 votes.
    if max_votes >= 2:
        # Find ALL players who received the max amount of votes (handles ties)
        eliminated_players = [p_idx for p_idx, count in vote_counts.items() if count == max_votes]
        
        # Check if any of the eliminated players ended the game as a Werewolf
        wolf_killed = any(end_roles[p_idx] == "Werewolf" for p_idx in eliminated_players)
        
        if wolf_killed:
            human_correct_werewolf_kills += 1
    else:
        # Everyone received 1 vote (Circle Vote)
        attempted_circle_votes += 1
        
        # It is only a CORRECT circle vote if there are no active Werewolves
        if "Werewolf" not in end_roles:
            human_correct_circle_votes += 1

# --- Print Results ---
total_human_wins = human_correct_werewolf_kills + human_correct_circle_votes
human_win_rate = (total_human_wins / valid_human_games) * 100 if valid_human_games > 0 else 0

print("-" * 45)
print("HUMAN BASELINE")
print("-" * 45)
print(f"Total Games in Index:      {total_games_in_index}")
print(f"Valid Human Games:         {valid_human_games}")
print(f"Correct Werewolf Kills:    {human_correct_werewolf_kills}")
print(f"Correct Circle Votes:      {human_correct_circle_votes}")
print(f"Total Attempted Circles:   {attempted_circle_votes}")
print("-" * 45)
print(f"Total Human Wins:          {total_human_wins}")
print(f"Human Win Rate:            {human_win_rate:.2f}%")

---------------------------------------------
HUMAN BASELINE
---------------------------------------------
Total Games in Index:      191
Valid Human Games:         171
Correct Werewolf Kills:    77
Correct Circle Votes:      7
Total Attempted Circles:   12
---------------------------------------------
Total Human Wins:          84
Human Win Rate:            49.12%


### Do humans and LLM fail on the same games?

In [14]:
both_correct = 0
both_wrong = 0
human_correct_llm_wrong = 0
llm_correct_human_wrong = 0

for res_file in result_files:
    with open(res_file, "r", encoding="utf-8") as f:
        res_data = json.load(f)
        
    session_name = res_data.get("session_name")
    game_key = res_data.get("game_key")
    
    gt_game = ground_truth_lookup.get((session_name, game_key))
    if not gt_game:
        continue
        
    # human votes
    raw_votes = gt_game.get("voting_outcome", [])
    
    # Skip games where human votes are missing
    if not raw_votes or "N/A" in raw_votes or "NA" in raw_votes or raw_votes == "N/A":
        continue
        
    human_won = False
    votes = [int(v) for v in raw_votes]
    vote_counts = Counter(votes)
    end_roles = gt_game.get("end_roles", [])
    
    if vote_counts:
        max_votes = max(vote_counts.values())
        if max_votes >= 2:
            eliminated_players = [p for p, count in vote_counts.items() if count == max_votes]
            human_won = any(end_roles[p] == "Werewolf" for p in eliminated_players)
        else:
            # Human circle vote
            human_won = ("Werewolf" not in end_roles)

    # llm wins
    validation_info = res_data.get("validation", {})
    chosen_vote = validation_info.get("chosen_vote")
    
    if not validation_info.get("is_valid") or not chosen_vote:
        continue 
        
    player_names = gt_game["player_names"]
    llm_won = False
    llm_valid_move = False
    
    if chosen_vote in player_names:
        llm_valid_move = True
        player_idx = player_names.index(chosen_vote)
        llm_won = (end_roles[player_idx] == "Werewolf")
        
    elif chosen_vote.strip().lower() == "circle vote":
        llm_valid_move = True
        llm_won = ("Werewolf" not in end_roles)
        
    if not llm_valid_move:
        continue 

    # differences
    if llm_won and human_won:
        both_correct += 1
    elif not llm_won and not human_won:
        both_wrong += 1
    elif human_won and not llm_won:
        human_correct_llm_wrong += 1
    elif llm_won and not human_won:
        llm_correct_human_wrong += 1

# stats
total_head_to_head = both_correct + both_wrong + human_correct_llm_wrong + llm_correct_human_wrong

print(f"Total Valid Overlapping Games:  {total_head_to_head}")
print(f"Both Correct:                   {both_correct}")
print(f"Both Wrong:                     {both_wrong}")
print(f"Human Correct, LLM Wrong:       {human_correct_llm_wrong}")
print(f"LLM Correct, Human Wrong:       {llm_correct_human_wrong}")
print("-" * 55)

# relative performance
if total_head_to_head > 0:
    llm_win_rate = ((both_correct + llm_correct_human_wrong) / total_head_to_head) * 100
    human_win_rate = ((both_correct + human_correct_llm_wrong) / total_head_to_head) * 100
    
    print(f"Human Win Rate (in overlap):    {human_win_rate:.2f}%")
    print(f"LLM Win Rate (in overlap):      {llm_win_rate:.2f}%")

Total Valid Overlapping Games:  147
Both Correct:                   36
Both Wrong:                     52
Human Correct, LLM Wrong:       34
LLM Correct, Human Wrong:       25
-------------------------------------------------------
Human Win Rate (in overlap):    47.62%
LLM Win Rate (in overlap):      41.50%
